# EchoFind: Hardware Acceleration Benchmark

This notebook demonstrates the performance of the **EchoFind Neural Sound Retrieval** pipeline. It measures the inference times for our two core deep learning models:
1. **LAION-CLAP** (`laion/clap-htsat-fused`) for multi-modal audio/text embedding.
2. **Faster-Whisper** (`tiny`/`base`) for fast speech-to-text timestamping.

### The Goal
We are applying for a **Hugging Face GPU Community Grant**. Currently, EchoFind is deployed on a free CPU tier, which creates a massive bottleneck when users upload audio to be indexed. This benchmark proves that hardware acceleration (such as an NVIDIA T4 or A10G, or Apple MPS locally) is absolutely necessary to achieve real-time audio indexing.

In [ ]:
import time
import os
import sys
import numpy as np
import librosa
import tempfile
import soundfile as sf
from pathlib import Path

# Ensure the backend source code is in the path to load the models
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "backend")))

from src.core.embedder import ClapEmbedder
from src.core.indexer import get_whisper_model

print("Libraries loaded successfully.")

## 1. Load Models into Memory
We initialize the CLAP Embedder (which automatically targets available GPU/MPS accelerators) and the Faster-Whisper model.

In [ ]:
print("[1/3] Loading Models into Memory...")

t0 = time.time()
embedder = ClapEmbedder()
t_clap_load = time.time() - t0
print(f"\u2705 LAION-CLAP loaded in {t_clap_load:.2f} seconds. (Device: {embedder.device})")

t0 = time.time()
whisper_model = get_whisper_model()
t_whisper_load = time.time() - t0
print(f"\u2705 Faster-Whisper loaded in {t_whisper_load:.2f} seconds.")

## 2. Generate Dummy Audio Data
We generate 60 seconds of white noise to simulate a standard user upload.

In [ ]:
duration_s = 60
sr = 48000
print(f"Generating {duration_s}s of dummy audio data at {sr}Hz...")
audio_data = np.random.uniform(-1, 1, size=(int(duration_s * sr),)).astype(np.float32)

# Chunk audio for CLAP (simulating the AudioFragmenter behavior: e.g., 2-second chunks)
chunk_size = sr * 2
chunks = [audio_data[i:i+chunk_size] for i in range(0, len(audio_data), chunk_size)]
print(f"Generated {len(chunks)} audio chunks (2 seconds each).")

## 3. Benchmark CLAP Embedding
We pass the chunks through the CLAP neural network to generate 512D acoustic vectors.

In [ ]:
print(f"[2/3] Benchmarking CLAP Embedding ({duration_s}s audio -> {len(chunks)} chunks)...")
t0 = time.time()

# Batch process in chunks of 16 (matching production indexer logic)
batch_size = 16
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    _ = embedder.embed_audio_batch(batch)
    
t_clap_infer = time.time() - t0
clap_rtf = t_clap_infer / duration_s

print(f"\u23F1\uFE0F CLAP Inference Time: {t_clap_infer:.2f} seconds")
print(f"\ud83d\udcca Real-Time Factor (RTF): {clap_rtf:.2f}x (Lower is better)")

## 4. Benchmark Whisper Transcription
We pass the full audio file through Faster-Whisper to generate timestamped speech-to-text data.

In [ ]:
print(f"[3/3] Benchmarking Whisper Transcription ({duration_s}s audio)...")

with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
    sf.write(tmp_file.name, audio_data, sr)
    tmp_path = tmp_file.name
    
try:
    t0 = time.time()
    segments, info = whisper_model.transcribe(tmp_path, word_timestamps=True, vad_filter=True)
    # Force evaluation of the generator to execute transcription
    list(segments)
    
    t_whisper_infer = time.time() - t0
    whisper_rtf = t_whisper_infer / duration_s
    
    print(f"\u23F1\uFE0F Whisper Inference Time: {t_whisper_infer:.2f} seconds")
    print(f"\ud83d\udcca Real-Time Factor (RTF): {whisper_rtf:.2f}x (Lower is better)")
finally:
    os.remove(tmp_path)

## 5. Conclusion & The Need for a GPU
The metrics below demonstrate the performance of the pipeline.

In [ ]:
total_time = t_clap_infer + t_whisper_infer
total_rtf = clap_rtf + whisper_rtf

print("="*50)
print("\ud83d\udcc8 BENCHMARK SUMMARY")
print("="*50)
print(f"Total Processing Time for {duration_s}s audio: {total_time:.2f} seconds")
print(f"Combined RTF: {total_rtf:.2f}x")

print("\n***\n")
if total_rtf < 0.2:
    print("\u2705 ACCELERATION DETECTED: This machine is using hardware acceleration (e.g., Apple MPS or NVIDIA CUDA).")
    print("Because of matrix multiplication acceleration, processing is incredibly fast.")
    print("\n\u26a0\ufe0f THE PROBLEM: When EchoFind is deployed on Hugging Face Spaces free tier, it falls back to standard CPUs without this acceleration.")
    print("This causes the pipeline to bottleneck severely, completely ruining the real-time user experience.")
    print("A GPU grant (such as T4 or A10G) is absolutely required to match this local accelerated performance in production.")
else:
    print("\u26a0\ufe0f WARNING: The system is running too slow for a seamless real-time user experience.")
    print("You are currently running on standard CPUs.")
    print("A GPU (e.g. T4 or A10G) would massively accelerate matrix multiplications for both transformers, bringing processing times down to fractions of a second.")